In [ ]:
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import re
from sqlalchemy.exc import OperationalError

# --- Database Connection Details ---
db_user = 'postgres'
db_password = ''
db_host = 'localhost'
db_port = '5432'
db_name = 'pt4'

# Create the database connection URL for SQLAlchemy
db_url = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# --- SQL Query ---
# Using SQLAlchemy's text() construct allows for safe parameter binding.
# The :player_id is a named parameter that we will supply later.
sql_query = text("""
                 SELECT s.date_played,
                        cl.amt_bb,
                        p.player_name as hero_name,
                        h.*
                 FROM cash_hand_histories AS h
                          JOIN cash_hand_player_statistics AS s ON h.id_hand = s.id_hand
                          JOIN cash_limit cl on s.id_limit = cl.id_limit
                          JOIN player p on s.id_player = p.id_player
                 WHERE s.id_player = :player_id
                   AND s.flg_f_saw
                   AND CAST(s.date_played AS DATE) = '2025-09-14'
                 order by id_hand desc;
                 """)

# --- Parameters ---
query_params = {"player_id": 93}  # <-- IMPORTANT: Replace 93 with your actual player ID

# --- Main Execution Block ---
hands_df = pd.DataFrame()  # Initialize an empty DataFrame

try:
    print("Creating database engine...")
    # Pass client_encoding directly to the DBAPI driver (psycopg2) via connect_args.
    # This is a cleaner, more direct way to handle encoding issues than environment variables.
    engine = create_engine(
        db_url,
        connect_args={'client_encoding': 'WIN1252'}
    )

    print("\nExecuting query and loading data into a DataFrame...")
    # Pandas can use the SQLAlchemy engine directly to manage connections.
    # We pass the query parameters separately for safety.
    hands_df = pd.read_sql_query(sql_query, engine, params=query_params)

    print("Successfully loaded data.")
    print(f"Found {len(hands_df)} hands.")

    # Overwrite id_hand with the value extracted from the history text
    if not hands_df.empty:
        print("\nExtracting hand ID from history text to overwrite 'id_hand' column...")
        # This regex finds the numeric ID from the first line of the hand history.
        # e.g., "PokerStars Hand #123456789: ..." -> "123456789"
        hands_df['id_hand'] = hands_df['history'].str.extract(r'Hand #(\d+)', expand=False)
        hands_df['id_hand'] = pd.to_numeric(hands_df['id_hand'], errors='coerce')

        # Drop rows where the hand ID could not be extracted
        hands_df.dropna(subset=['id_hand'], inplace=True)
        print("'id_hand' column has been updated.")

except OperationalError as e:
    print(f"CONNECTION FAILED: Could not connect to the database.", file=sys.stderr)
    print(f"Please check your credentials and ensure the PostgreSQL server is running.", file=sys.stderr)
    print(f"Error details: {e}", file=sys.stderr)
except Exception as e:
    print(f"\nQUERY FAILED: An unexpected error occurred.", file=sys.stderr)
    print(f"Error details: {e}", file=sys.stderr)

In [ ]:
# hands_df = hands_df[(hands_df['id_hand'] == 2546893392)]
hands_df = hands_df[hands_df['history'].str.contains(' RIVER ')]
hands_df

In [ ]:
import re

# The previous cell loaded data into 'hands_df'. This cell will process it.

def convert_history_to_bb(row):
    """
    Parses the 'history' text and converts all dollar amounts to Big Blinds.
    This function is designed to be used with pandas.DataFrame.apply().
    """
    history_text = row['history']
    big_blind_cents = row['amt_bb']

    # Return original text if there's no history or BB amount is invalid
    if pd.isna(history_text) or not big_blind_cents or big_blind_cents <= 0:
        return history_text

    # Correcting a bug: amt_bb is in cents, so it must be divided by 100.
    bb_dollars = big_blind_cents

    def replacer(match):
        """A nested function to replace each matched dollar amount."""
        # The captured group is the numeric value (e.g., '100.00' from '$100.00')
        dollar_amount = float(match.group(1))

        bb_amount = dollar_amount / bb_dollars
        # Return the amount in BB, formatted to two decimal places
        return f"{bb_amount:.2f} BB"

    # This regex finds dollar amounts (e.g., $100.00, $5.50, $0.50).
    # It looks for a '$' followed by digits and an optional decimal part.
    # The parentheses create a capturing group for the numeric value.
    pattern = re.compile(r'\$(\d+\.?\d*)')

    return pattern.sub(replacer, history_text)


# --- Main Processing Block ---

# Check if the required columns exist. Your query `SELECT h.*` should provide 'history' and 'amt_bb'.
if 'history' in hands_df.columns and 'amt_bb' in hands_df.columns:
    print("Parsing hand histories to convert dollar amounts to Big Blinds...")
    hands_df['history_bb'] = hands_df.apply(convert_history_to_bb, axis=1)
    print("Conversion complete.")

    print("\nDataFrame Head with BB Conversion:")
    display(hands_df[['history', 'history_bb', 'amt_bb']].head())
else:
    print("\nERROR: Could not perform conversion.", file=sys.stderr)
    print("The DataFrame must contain 'history' and 'amt_bb' columns.", file=sys.stderr)
    print(f"Available columns: {hands_df.columns.tolist()}", file=sys.stderr)
hands_df

In [ ]:
import numpy as np

def analyze_hand(row):
    """
    Analyzes a hand history row to extract key information in a single pass.
    This is far more efficient than parsing the same text multiple times.
    """
    history_text = row['history_bb']
    hero_name = row['hero_name']

    if pd.isna(history_text):
        return None

    lines = history_text.split('\n')
    if len(lines) < 2:
        return None  # Not a valid hand history

    # 1. Find the button seat from the second line of the hand history
    button_line = lines[1]
    button_match = re.search(r'Seat #(\d+) is the button', button_line)
    if not button_match:
        return None  # Could not determine button seat
    button_seat = int(button_match.group(1))

    # 2. Map all active players to their seats and get their stacks
    player_stacks = {}
    seat_player_map = {}
    seat_pattern = re.compile(r"Seat (\d+): (.+?) \((\d+\.?\d*) BB\)(?!.*is sitting out)")
    for match in seat_pattern.finditer(history_text):
        seat_num = int(match.group(1))
        player_name = match.group(2).strip()
        stack_size = float(match.group(3))

        player_stacks[player_name] = stack_size
        seat_player_map[seat_num] = player_name

    if not player_stacks or hero_name not in player_stacks:
        return None

    # 3. Calculate player positions based on the button's seat number
    active_seats = sorted(seat_player_map.keys())
    num_players = len(active_seats)
    player_positions = {}

    if num_players < 2:
        return None  # Not enough players for a game

    try:
        button_index = active_seats.index(button_seat)
    except ValueError:
        return None  # Button player is not in the list of active players

    # Assign positions relative to the button, handling different table sizes
    if num_players == 2:
        player_positions[seat_player_map[active_seats[button_index]]] = 'BTN'  # In HU, BTN is SB
        player_positions[seat_player_map[active_seats[(button_index + 1) % num_players]]] = 'BB'
    else:
        # For 3+ players
        player_positions[seat_player_map[active_seats[button_index]]] = 'BTN'
        player_positions[seat_player_map[active_seats[(button_index + 1) % num_players]]] = 'SB'
        player_positions[seat_player_map[active_seats[(button_index + 2) % num_players]]] = 'BB'
        if num_players > 3:
            player_positions[seat_player_map[active_seats[(button_index - 1 + num_players) % num_players]]] = 'CO'
        if num_players > 4:
            player_positions[seat_player_map[active_seats[(button_index - 2 + num_players) % num_players]]] = 'MP'
        if num_players > 5:
            player_positions[seat_player_map[active_seats[(button_index - 3 + num_players) % num_players]]] = 'UTG'

    # This regex is designed to be flexible. It captures the player, action, and an optional
    # BB amount that can appear anywhere after the action (e.g., "raises 2.80 BB", "calls 4.60 BB").
    action_pattern = re.compile(r"(.+?):? (raises|bets|calls|checks|folds)(?:.*? ([\d\.]+) BB)?(.*)")

    def _parse_street_actions(street_text, known_players):
        """Helper to parse actions from a block of text for a single street."""
        actions = []
        if not street_text:
            return actions
        for match in action_pattern.findall(street_text):
            player_name = match[0].strip()
            if player_name.endswith(':'):
                player_name = player_name[:-1].strip()

            # Skip system messages like "Uncalled bet..." that don't have a known player
            if player_name not in known_players:
                continue

            actions.append({
                'player': player_name,
                'action': match[1],
                'amount': float(match[2]) if match[2] else 0,
                'is_all_in': 'is all-in' in match[3].strip()
            })
        return actions

    # Split the history into sections based on the '***' dividers
    # The regex uses a lookahead to keep the dividers as part of the split result.
    street_sections = re.split(r'(?=\*\*\* (?:FLOP|TURN|RIVER|SHOW DOWN|SUMMARY) \*\*\*)', history_text)

    preflop_text = street_sections[0]
    flop_text = ""
    turn_text = ""
    river_text = ""

    # A hand might not have all streets, so we search for each one.
    for section in street_sections[1:]:
        if section.startswith('*** FLOP ***'):
            flop_text = section
        elif section.startswith('*** TURN ***'):
            turn_text = section
        elif section.startswith('*** RIVER ***'):
            river_text = section

    preflop_actions = _parse_street_actions(preflop_text, player_stacks)
    flop_actions = _parse_street_actions(flop_text, player_stacks)
    turn_actions = _parse_street_actions(turn_text, player_stacks)
    river_actions = _parse_street_actions(river_text, player_stacks)

    # 5. Determine players at flop from the canonical action list, ensuring consistency.
    players_who_folded = {act['player'] for act in preflop_actions if act['action'] == 'folds'}
    players_at_start = set(player_stacks.keys())
    players_who_saw_flop = players_at_start - players_who_folded

    # Create a dictionary of positions for only the players who saw the flop.
    active_player_positions = {p: player_positions[p] for p in players_who_saw_flop if p in player_positions}

    # 6. Parse board cards
    board_cards = {}
    flop_match = re.search(r'\*\*\* FLOP \*\*\* \[(.+?)\]', history_text)
    if flop_match:
        board_cards['flop'] = flop_match.group(1).split(' ')

    turn_match = re.search(r'\*\*\* TURN \*\*\* .*?\[([A-Za-z0-9]{2})\]', history_text)
    if turn_match:
        board_cards['turn'] = turn_match.group(1)

    river_match = re.search(r'\*\*\* RIVER \*\*\* .*?\[([A-Za-z0-9]{2})\]', history_text)
    if river_match:
        board_cards['river'] = river_match.group(1)

    # 7. Extract Hero's cards
    hero_cards_match = re.search(rf"Dealt to {re.escape(hero_name)} \[([A-Za-z0-9\s]+)\]", history_text)
    hero_cards = None
    if hero_cards_match:
        # e.g., "Ac 7c" -> "Ac7c"
        hero_cards = hero_cards_match.group(1).replace(' ', '')

    return {
        "players_at_flop": len(players_who_saw_flop),
        "active_players": list(players_who_saw_flop),
        "active_player_positions": active_player_positions,
        "player_stacks": player_stacks,
        "player_positions": player_positions,
        "preflop_actions": preflop_actions,
        "flop_actions": flop_actions,
        "turn_actions": turn_actions,
        "river_actions": river_actions,
        "board_cards": board_cards,
        "hero_cards": hero_cards,
    }


print("Analyzing all hand histories...")
hands_df['analysis'] = hands_df.apply(analyze_hand, axis=1)
hands_df.dropna(subset=['analysis'], inplace=True)  # Drop hands that couldn't be parsed
print("Analysis complete.")

# --- Filter for Heads-Up (2-Player) Flops ---
initial_hand_count = len(hands_df)
hands_df = hands_df[hands_df['analysis'].apply(lambda x: x['players_at_flop'] == 2)].copy()
print(f"\nFiltered for heads-up pots. Kept {len(hands_df)} of {initial_hand_count} hands.")
hands_df

In [ ]:
import os
import re

RANGE_FOLDER = '6max_range'
CACHED_RANGES = {}  # Cache ranges to avoid reading the same file multiple times


def find_closest_size_dir(parent_path, target_size_bb):
    """Finds the subdirectory in parent_path that is numerically closest to the target bet size."""
    if not os.path.isdir(parent_path):
        return None

    size_dirs = {}
    for subdir in os.listdir(parent_path):
        # Check if the entry is a directory and matches the bet size format (e.g., "2.5bb")
        if os.path.isdir(os.path.join(parent_path, subdir)):
            match = re.match(r'(\d+\.?\d*)bb', subdir)
            if match:
                size_dirs[float(match.group(1))] = subdir

    if not size_dirs:
        return None

    # Find the key (size) in the dictionary that is closest to the target size
    closest_size = min(size_dirs.keys(), key=lambda k: abs(k - target_size_bb))
    return size_dirs[closest_size]


def load_range_from_file(filepath):
    """Loads a hand range from a file path and caches the result."""
    if not filepath:
        return None
    if filepath in CACHED_RANGES:
        return CACHED_RANGES[filepath]

    try:
        with open(filepath, 'r') as f:
            range_text = f.read().strip()
            CACHED_RANGES[filepath] = range_text
            return range_text
    except FileNotFoundError:
        # Cache the failure so we don't try again
        CACHED_RANGES[filepath] = None
        return None


def assign_ranges(row):
    """Dynamically builds a path based on pre-flop actions to find and load the correct solver ranges."""
    analysis = row['analysis']
    amt_bb = row['amt_bb']

    if not analysis or analysis['players_at_flop'] != 2:
        return None, None

    players = analysis['active_players']
    positions = analysis['player_positions']

    # Determine who is In Position (IP) and Out of Position (OOP)
    # The player who acts last post-flop is in position.
    p1, p2 = players[0], players[1]
    p1_pos, p2_pos = positions.get(p1), positions.get(p2)

    post_flop_order = ['SB', 'BB', 'UTG', 'MP', 'CO', 'BTN']
    try:
        if post_flop_order.index(p1_pos) > post_flop_order.index(p2_pos):
            ip_player, oop_player = p1, p2
        else:
            ip_player, oop_player = p2, p1
    except (ValueError, TypeError):
        # This can happen if a position is not in our standard list
        return None, None

    ip_pos = positions.get(ip_player)
    oop_pos = positions.get(oop_player)

    # --- Build path based on action sequence ---
    current_path = RANGE_FOLDER

    # First, identify players who have voluntarily put money in the pot (VPIP).
    # A fold is only a significant path segment if the player was already "in".
    vpip_players = {
        act['player'] for act in analysis['preflop_actions']
        if act['action'] in ['raises', 'bets', 'calls']
    }

    for act in analysis['preflop_actions']:
        player_pos = positions.get(act['player'])
        if not player_pos:
            continue

        # LOGIC CHANGE: Skip passive folds that don't create a new branch in the solver tree.
        if act['action'] == 'folds' and act['player'] not in vpip_players:
            continue

        # Append player position
        next_path_segment = os.path.join(current_path, player_pos)
        if not os.path.isdir(next_path_segment):
            return None, None
        current_path = next_path_segment

        # Append action description
        action = act['action']
        action_dir = None
        if action in ['raises', 'bets']:
            if act['is_all_in']:
                action_dir = 'AllIn'
            else:
                bet_size_bb = act['amount'] / amt_bb
                action_dir = find_closest_size_dir(current_path, bet_size_bb)
        elif action in ['calls', 'folds', 'checks']:
            # Correctly map plural actions ('calls', 'folds') to singular directory names ('Call', 'Fold')
            action_dir = action.rstrip('s').capitalize()

        if not action_dir:
            return None, None

        next_path_segment = os.path.join(current_path, action_dir)
        if not os.path.isdir(next_path_segment):
            return None, None
        current_path = next_path_segment

    # --- Load ranges from the final path ---
    ip_range_file = os.path.join(current_path, f"{ip_pos}_range.txt")
    oop_range_file = os.path.join(current_path, f"{oop_pos}_range.txt")

    ip_range = load_range_from_file(ip_range_file)
    oop_range = load_range_from_file(oop_range_file)

    return ip_range, oop_range


print("\nAssigning hand ranges based on pre-flop action...")
hands_df[['ip_range', 'oop_range']] = hands_df.apply(assign_ranges, axis=1).apply(pd.Series)
print("Range assignment complete.")

print("\nDataFrame with Hand Ranges:")
display(hands_df[['history', 'ip_range', 'oop_range']].head())

TODO: find SB calling ranges - For now removing 0 columns

In [ ]:
# --- Filter out hands where ranges could not be found ---
print(f"\nNumber of hands before filtering out null ranges: {len(hands_df)}")

# Keep rows where both ip_range and oop_range were successfully assigned.
hands_df = hands_df[hands_df['ip_range'].notna() & hands_df['oop_range'].notna()].copy()

print(f"Number of hands after filtering: {len(hands_df)}")

print("\nDataFrame Head after filtering for valid ranges:")
display(hands_df[['history', 'ip_range', 'oop_range']].head())

In [ ]:
import os
import math

# --- Create Directories ---
input_dir = 'C:\\Users\\Braindead\\PyCharmMiscProject\\input'
output_dir = 'C:\\Users\\Braindead\\PyCharmMiscProject\\output'
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


def generate_solver_file(row):
    """
    Generates a Texas Solver input file for a single hand history,
    using pre-computed data from the 'analysis' column.
    """
    try:
        # --- 1. Extract Data from the Row ---
        analysis = row['analysis']
        history_bb = row['history_bb']  # Use BB history for parsing blinds/board
        ip_range = row['ip_range']
        oop_range = row['oop_range']
        id_hand = row['id_hand']

        # --- 2. Determine IP and OOP Players (no change from original) ---
        players = analysis['active_players']
        positions = analysis['player_positions']
        p1, p2 = players[0], players[1]
        p1_pos, p2_pos = positions.get(p1), positions.get(p2)

        post_flop_order = ['SB', 'BB', 'UTG', 'MP', 'CO', 'BTN']
        if post_flop_order.index(p1_pos) > post_flop_order.index(p2_pos):
            ip_player, oop_player = p1, p2
        else:
            ip_player, oop_player = p2, p1

        # --- 3. Calculate Pot and Effective Stack at the START of the Flop ---
        # Determine pre-flop investment for each player to calculate the pre-rake pot
        player_investments = {}

        # Parse blinds from the BB history text (as this is not in 'analysis' yet)
        blind_pattern = re.compile(r"(.+?):? posts the (small|big) blind ([\d\.]+) BB")
        for match in blind_pattern.finditer(history_bb):
            player = match.group(1).strip()
            amount = float(match.group(3))
            player_investments[player] = amount

        # Process pre-flop actions from 'analysis' to find total investment per player
        current_bet_level = max(player_investments.values()) if player_investments else 0
        for act in analysis['preflop_actions']:
            player = act['player']
            action = act['action']
            if action == 'raises':
                amount = act['amount']
                player_investments[player] = amount
                current_bet_level = amount
            elif action == 'calls':
                player_investments[player] = current_bet_level
            elif action == 'bets':  # Handles open-limps
                amount = act['amount']
                player_investments[player] = amount
                current_bet_level = amount

        # The total pre-rake pot is the sum of all investments
        flop_pot_bb = sum(player_investments.values())

        # Calculate effective stack at the start of the flop
        p1_start_stack_bb = analysis['player_stacks'][p1]
        p2_start_stack_bb = analysis['player_stacks'][p2]

        p1_investment = player_investments.get(p1, 0)
        p2_investment = player_investments.get(p2, 0)

        p1_stack_at_flop = p1_start_stack_bb - p1_investment
        p2_stack_at_flop = p2_start_stack_bb - p2_investment

        effective_stack_bb = min(p1_stack_at_flop, p2_stack_at_flop)

        # --- 4. Parse Board Cards (but only provide the flop to the solver) ---
        flop_match = re.search(r"\*\*\* FLOP \*\*\* \[(.+?)\]", history_bb)
        if not flop_match:
            return False

        board = flop_match.group(1).replace(' ', ',')

        # --- 5. Define Bet Sizes for the Game Tree ---
        standard_bet_sizes = {33, 66}
        custom_bet_sizes = set()

        pot_on_street = flop_pot_bb
        street_actions_list = [
            analysis.get('flop_actions', []),
            analysis.get('turn_actions', []),
            analysis.get('river_actions', [])
        ]

        for street_actions in street_actions_list:
            if not street_actions:
                continue

            pot_for_this_street_actions = pot_on_street
            for act in street_actions:
                if act['action'] in ['bets', 'raises']:
                    bet_amount_bb = act['amount']
                    if pot_for_this_street_actions > 0:
                        bet_pct = (bet_amount_bb / pot_for_this_street_actions) * 100

                        is_significantly_different = True
                        for std_size in standard_bet_sizes:
                            if abs(bet_pct - std_size) / std_size <= 0.25:
                                is_significantly_different = False
                                break

                        if is_significantly_different:
                            rounded_pct = int(round(bet_pct, -1))
                            if rounded_pct > 0:
                                custom_bet_sizes.add(rounded_pct)

                if act['action'] in ['bets', 'calls', 'raises']:
                    pot_for_this_street_actions += act['amount']

            money_in_street = sum(
                act['amount'] for act in street_actions if act['action'] in ['bets', 'calls', 'raises'])
            pot_on_street += money_in_street

        final_bet_sizes = sorted(list(standard_bet_sizes.union(custom_bet_sizes)))

        # --- 6. Assemble the Solver Input File ---
        solver_input_template = f"""set_pot {flop_pot_bb:.2f}
set_effective_stack {effective_stack_bb:.2f}
set_board {board}
set_range_ip {ip_range}
set_range_oop {oop_range}
"""

        # Add the bet size lines for all streets and actions
        bet_sizes_str = ",".join(map(str, final_bet_sizes)) if final_bet_sizes else "33,66"
        streets_to_set = ['flop', 'turn', 'river']
        actions_to_set = ['bet', 'raise']

        for street in streets_to_set:
            for action in actions_to_set:
                solver_input_template += f"set_bet_sizes oop,{street},{action},{bet_sizes_str}\n"
                solver_input_template += f"set_bet_sizes ip,{street},{action},{bet_sizes_str}\n"

        # Add the special 'donk' bet for OOP on the river
        solver_input_template += f"set_bet_sizes oop,river,donk,{bet_sizes_str}\n"

        # Add the static solver commands
        solver_input_template += f"""set_allin_threshold 1.0
set_raise_limit 3
build_tree
set_thread_num 8
set_accuracy 0.5
set_max_iteration 200
set_print_interval 10
set_use_isomorphism 1
set_hand_analysis 1
start_solve
set_dump_rounds 3
dump_result {os.path.join(output_dir, f'{id_hand}.json')}
"""

        # --- 7. Write the File ---
        input_filepath = os.path.join(input_dir, f"{id_hand}.txt")
        with open(input_filepath, 'w') as f:
            f.write(solver_input_template)

        return True

    except Exception as e:
        print(f"Failed to process hand {row.get('id_hand', 'N/A')}: {e}", file=sys.stderr)
        return False


# --- Main Execution Block ---
print(f"\nGenerating solver input files for {len(hands_df)} hands...")

if not hands_df.empty:
    success_count = hands_df.apply(generate_solver_file, axis=1).sum()
    print(
        f"Successfully generated {success_count} of {len(hands_df)} solver input files in the '{input_dir}\\' directory.")
else:
    print("No hands to process.")

hands_df

In [ ]:
import subprocess
import os
from tqdm import tqdm

# --- Configuration ---
# Paths to the solver executable and its resources directory.
# Using raw strings (r'...') is a good practice for Windows paths.
SOLVER_EXE_PATH = r'C:\Users\Braindead\git\TexasSolver\cmake-build-debug\TexasSolverCli.exe'
RESOURCE_DIR_PATH = r'C:\Users\Braindead\git\TexasSolver\resources'

# The input directory where .txt files were generated in the previous cell.
# This should match the 'input_dir' variable from the previous cell.
INPUT_DIR = r'C:\Users\Braindead\PyCharmMiscProject\input'

# Number of parallel solver processes to run, as requested.
MAX_WORKERS = 4


def run_solver_command(command: list) -> (str, bool, str):
    """
    Executes a TexasSolver command-line tool command.

    Args:
        command: A list of strings representing the command to execute.

    Returns:
        A tuple containing: (hand_id_str, success_flag, message).
    """
    # Extract hand_id from command for logging purposes
    try:
        input_file_path = command[command.index('--input_file') + 1]
        hand_id = os.path.basename(input_file_path).split('.')[0]
    except (ValueError, IndexError):
        hand_id = "Unknown"

    # Print the command that will be executed.
    print(f"Executing for hand {hand_id}: {' '.join(command)}", flush=True)

    try:
        # Execute the command. We capture the output to check for errors.
        # A timeout is added to prevent processes from hanging indefinitely.
        subprocess.run(
            command,
            check=True,  # Raise an exception for non-zero exit codes
            capture_output=True,
            text=True,  # Decode stdout/stderr as text
            timeout=600  # 10-minute timeout per hand
        )
        return hand_id, True, "Solver completed successfully."
    except subprocess.CalledProcessError as e:
        # This occurs if the solver returns a non-zero exit code (i.e., an error).
        command_str = ' '.join(e.cmd)
        error_message = (
            f"Solver failed with exit code {e.returncode}.\n"
            f"COMMAND: {command_str}\n"
            f"STDOUT:\n{e.stdout}\n"
            f"STDERR:\n{e.stderr}"
        )
        return hand_id, False, error_message
    except subprocess.TimeoutExpired as e:
        command_str = ' '.join(e.cmd)
        error_message = (
            f"Solver process timed out after {e.timeout} seconds.\n"
            f"COMMAND: {command_str}\n"
            f"STDOUT:\n{e.stdout or ''}\n"
            f"STDERR:\n{e.stderr or ''}"
        )
        return hand_id, False, error_message
    except Exception as e:
        return hand_id, False, f"An unexpected error occurred: {e}"


# --- Main Execution Block ---
if 'hands_df' not in globals() or hands_df.empty:
    print("ERROR: `hands_df` is not defined or is empty. Please run the previous cells first.")
else:
    print(f"Preparing to run solver for {len(hands_df)} hands...")

    # 1. Create a full list of commands to execute
    commands_to_run = []
    for _, row in hands_df.iterrows():
        hand_id = row['id_hand']
        input_file = os.path.join(INPUT_DIR, f"{hand_id}.txt")
        if os.path.exists(input_file):
            command = [
                SOLVER_EXE_PATH,
                '--input_file', input_file,
                '--resource_dir', RESOURCE_DIR_PATH
            ]
            commands_to_run.append(command)
        else:
            print(f"WARNING: Input file for hand {hand_id} not found. Skipping.")

    print(f"Starting solver for {len(commands_to_run)} hands sequentially...")

    # 2. Run commands sequentially for debugging and clearer output
    success_count = 0
    failure_count = 0
    for command in tqdm(commands_to_run, desc="Processing hands sequentially"):
        hand_id, success, message = run_solver_command(command)
        if success:
            success_count += 1
            print(f"Hand {hand_id}: SUCCESS.")
        else:
            failure_count += 1
            print(f"Hand {hand_id}: FAILED. Reason: {message}")

    print("\n--- Solver Execution Summary ---")
    print(f"Successfully processed: {success_count}")
    print(f"Failed to process:     {failure_count}")
    print("--------------------------------")

In [ ]:
import json
import re
from typing import List, Dict, Any


def load_json_data(filepath: str) -> Dict[str, Any]:
    """Loads the solver's JSON output file."""
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{filepath}' was not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: The file '{filepath}' is not a valid JSON file.")
        return None


def find_closest_action(available_actions: List[str], target_action: str) -> str:
    """
    Finds the closest matching action from a list of available actions.
    For 'CALL' or 'CHECK', it requires an exact match.
    For 'BET' or 'RAISE', it finds the numerically closest size.
    """
    if target_action in ['CALL', 'CHECK', 'FOLD']:
        return target_action if target_action in available_actions else None

    target_match = re.match(r'(BET|RAISE) (\d+),000000', target_action)
    if not target_match:
        return None

    target_type = target_match.group(1)
    target_amount = int(target_match.group(2))

    sized_actions = {}
    for action in available_actions:
        action_match = re.match(rf'{target_type} (\d+),000000', action)
        if action_match:
            amount = int(action_match.group(1))
            sized_actions[action] = amount

    if not sized_actions:
        return None

    closest_action = min(
        sized_actions.keys(),
        key=lambda k: abs(sized_actions[k] - target_amount)
    )
    return closest_action


def get_hand_strategy(node: Dict[str, Any], hand: str) -> Dict[str, float]:
    """
    Analyzes and returns the strategy for a specific hand at a given node.
    """
    if not node or node.get('node_type') != 'action_node':
        return None

    strategy_data = node.get('strategy')
    if not strategy_data:
        return None

    actions = strategy_data.get('actions')
    hand_strategies = strategy_data.get('strategy')

    if not actions or not hand_strategies:
        return None

    hand_reversed = hand[2:] + hand[:2]
    strategy_values = None
    if hand in hand_strategies:
        strategy_values = hand_strategies[hand]
    elif hand_reversed in hand_strategies:
        strategy_values = hand_strategies[hand_reversed]

    if strategy_values is None:
        return None

    strategy_dict = {}
    for i, action in enumerate(actions):
        clean_action = action.replace(',', '').replace('000000', '')
        probability = strategy_values[i]
        strategy_dict[clean_action] = probability

    return strategy_dict


def action_to_path_string(act: Dict[str, Any]) -> str:
    """Converts an action dictionary from analysis into a solver-compatible path string."""
    action = act['action'].upper()
    if action in ['BETS', 'RAISES']:
        amount = int(act['amount'])
        return f"{action.rstrip('S')} {amount},000000"
    elif action == 'CALLS':
        return 'CALL'
    elif action == 'CHECKS':
        return 'CHECK'
    elif action == 'FOLDS':
        return 'FOLD'
    return None


def analyze_hero_play_vs_solver(row: pd.Series) -> List[Dict[str, Any]]:
    """
    Analyzes a single hand (row) against its corresponding GTO solver output.
    This function traverses the game tree based on the hand history and records
    the GTO strategy at each of the hero's decision points.
    """
    hand_id = row['id_hand']
    analysis_data = row['analysis']
    hero_name = row['hero_name']

    solver_filepath = f'C:/Users/Braindead/PyCharmMiscProject/output/{hand_id}.json'
    solver_tree = load_json_data(solver_filepath)
    if not solver_tree:
        print(f"Warning: Could not load solver file for hand {hand_id}. Skipping analysis for this hand.")
        return None

    hero_cards = analysis_data.get('hero_cards')
    if not hero_cards:
        print(f"Warning: Could not determine hero's cards for hand {hand_id}. Skipping.")
        return None

    # --- Build a single path of actions and cards for traversal ---
    # --- Build a single path of actions and cards for traversal ---
    path_components = []
    flop_actions = analysis_data.get('flop_actions', [])
    for act in flop_actions:
        path_components.append({'type': 'action', 'data': act, 'street': 'Flop'})
 
    turn_card = analysis_data.get('board_cards', {}).get('turn')
    if turn_card:
        path_components.append({'type': 'card', 'data': turn_card, 'street': 'Turn'})
        turn_actions = analysis_data.get('turn_actions', [])
        for act in turn_actions:
            path_components.append({'type': 'action', 'data': act, 'street': 'Turn'})
 
    river_card = analysis_data.get('board_cards', {}).get('river')
    if river_card:
        path_components.append({'type': 'card', 'data': river_card, 'street': 'River'})
        river_actions = analysis_data.get('river_actions', [])
        for act in river_actions:
            path_components.append({'type': 'action', 'data': act, 'street': 'River'})
 
    # --- Traverse the tree once, collecting hero's actions ---
    current_node = solver_tree
    hero_actions_summary = []
 
    for component in path_components:
        if not current_node:
            # This can happen if the path is invalid or the tree is corrupt.
            # We stop further analysis for this hand but keep what we have.
            break
 
        component_type = component['type']
        
        if component_type == 'action':
            act = component['data']
            
            # If it's the hero's turn, analyze the decision before moving to the next node
            if act['player'] == hero_name:
                gto_strategy = get_hand_strategy(current_node, hero_cards)
                action_str = action_to_path_string(act)
                if action_str:
                    clean_action = action_str.replace(',', '').replace('000000', '')
                    hero_actions_summary.append({
                        'street': component['street'],
                        'hand': hero_cards,
                        'player_action': clean_action,
                        'gto_strategy': gto_strategy
                    })
 
            # Now, traverse to the next node based on the action taken
            action_path_str = action_to_path_string(act)
            if not action_path_str:
                continue
 
            node_type = current_node.get('node_type')
            if node_type == 'action_node':
                children = current_node.get('childrens', {})
                if action_path_str in children:
                    current_node = children[action_path_str]
                else:
                    closest_action = find_closest_action(list(children.keys()), action_path_str)
                    if closest_action:
                        current_node = children[closest_action]
                    else:
                        current_node = None
            else:
                current_node = None # Invalid state
 
        elif component_type == 'card':
            card = component['data']
            node_type = current_node.get('node_type')
            if node_type == 'chance_node':
                dealcards = current_node.get('dealcards', {})
                current_node = dealcards.get(card)
            else:
                current_node = None # Invalid state
 
    return hero_actions_summary if hero_actions_summary else None


def main():
    """
    Main function to run the GTO analysis for all hands in the DataFrame
    and store the results in a new column.
    """
    if 'hands_df' not in globals() or hands_df.empty:
        print("Error: `hands_df` is not defined or is empty. Please run the previous cells.")
        return

    print(f"Starting GTO analysis for {len(hands_df)} hand(s)...")

    # Apply the analysis function to each row and store the result in a new column
    hands_df['gto_analysis'] = hands_df.apply(analyze_hero_play_vs_solver, axis=1)

    print("GTO analysis complete.")

    # Display a summary of the results
    successful_analyses = hands_df['gto_analysis'].notna().sum()
    print(f"Successfully analyzed {successful_analyses} of {len(hands_df)} hands.")

    print("\nDataFrame with GTO analysis results:")
    # Show relevant columns, including the new one
    display(hands_df[['id_hand', 'hero_name', 'hero_cards', 'gto_analysis']].head())

if __name__ == '__main__':
    main()
